# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAIRT

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Qualcomm AI Runtime (QAIRT) SDK**.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to QAIRT's Deep Learning Container (DLC) format using `qairt-converter`.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference using `qairt-quantizer`.
6. **HTP context binary compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325) using `qairt-context-binary-generator`.

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import glob
import os
import random
import torch
import uuid

import numpy as np

from libreyolo import LibreYOLO
from PIL import Image

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (resized to 640×640, normalized to `[0, 1]`).
- A **representative calibration dataset** in `.raw` binary format. QAIRT's `qairt-quantizer` uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `preprocess()` helper function used throughout this pipeline.

In [2]:
def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    Args:
        original_image (np.ndarray): Input image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image normalized to [0, 1].
    """

    image = Image.fromarray(original_image).convert("RGB")
    image = image.resize((size, size), Image.BILINEAR)

    image = np.asarray(image).astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    image = np.expand_dims(image, axis=0)   # CHW -> NCHW

    return image

### Preparing the Calibration Dataset

QAIRT's quantization tool (`qairt-quantizer`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(H, W, C)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **100 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `qairt-quantizer`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 100

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        image = Image.open(filename)
        original_image = np.array(image)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

QAIRT does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which `qairt-converter` can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model produces **three output heads** corresponding to different detection scales:
- `output_small` (80×80) — detects small objects.
- `output_medium` (40×40) — detects medium objects.
- `output_large` (20×20) — detects large objects.

In [4]:
if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

yolo = LibreYOLO("../models/LibreYOLOXs.pt")
torch_model = yolo.model
torch_model.eval()

if not os.path.exists("../models/LibreYOLOXs.onnx"):
    dummy = torch.randn(1, 3, 640, 640)
    torch.onnx.export(
        torch_model,
        dummy,
        "../models/LibreYOLOXs.onnx",
        opset_version=18,
        dynamo=False,
        input_names=["images"],
        output_names=[
            "output_small",   # 80x80
            "output_medium",  # 40x40
            "output_large"    # 20x20
        ]
    )

## Converting the Model to QAIRT DLC Format

QAIRT uses the **DLC (Deep Learning Container)** format as an intermediate representation. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `qairt-converter` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [5]:
!bash -c 'qairt-converter \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/qairt/LibreYOLOXs_fp32.dlc'

2026-05-09 14:15:07,885 - 278 - INFO - INFO_INITIALIZATION_SUCCESS: 
2026-05-09 14:15:07,929 - 283 - WARNING - Unable to register converter supported Operation [Inverse:Version 1] with your Onnx installation. Converter will bail if Model contains this Op.
2026-05-09 14:15:08,008 - 283 - WARNING - --desired_input_shape and -d are deprecated. Use --source_model_input_shape or -s for achieving this functionality
2026-05-09 14:15:08,008 - 278 - INFO - Input shape info 
2026-05-09 14:15:09,554 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-09 14:15:09,554 - 283 - WARNING - Simplified model validation failed
2026-05-09 14:15:09,773 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-09 14:15:09,774 - 283 - WARNING - Simplified model validation failed
2026-05-09 14:15:09,953 - 283 - WARNING - Unused Input nodes found: []
2026-05-09 14:15:10,460 - 283 - WARNING - Unused Input nodes found: []
2026-05-09 14:15

### Inspecting the FP32 DLC

The `qairt-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by QAIRT before proceeding to quantization.

In [6]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; validate_models=False; use_quantize_v2=False; use_onnx_relay=False; unroll_lstm_time_steps=True; unroll_gru_time_steps=True; transforms_metadata=None; tf_summary=False; skip_validation=False; signature_name=; dump_value_info=False; remove_unused_inputs=False; expand_sparse_op_structure=False; quantizer_log_level=LogLevel.NONE; quantization_overrides=; enable_match_topk=False; pytorch_custom_op_lib=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=False; define_symbol=None; use_convert_quantization_nodes=False; packed_max_seq=1; packed_masked_softmax_inputs=[]; output_layout=[]; float_bias_bitwidth=0; show_unconsumed_nodes=False; out_names=[]; float_bitwidth=32; optimization_pass_mode=ir_optimizer_disable; apply_masked_softmax=uncompressed; en

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `qairt-quantizer` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [7]:
!bash -c 'qairt-quantizer \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

     2.0ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/f4226a75-43c8-40bc-956e-f80daf3b4654.raw
raw/b1e79729-2b54-4a2e-a5a6-8ca7f7470b51.raw
raw/0e040cc5-2a5f-411a-aa4b-7d5c27b57249.raw
raw/d51e7430-77ec-490d-bfa4-511f86581cd7.raw
raw/ca3edc56-d546-4d9e-acbf-57e001d58e30.raw
raw/5f1bbf7b-ae92-4cbf-bd48-5295395c0114.raw
raw/32f19940-805d-4f92-92ff-c39c67ec5d5a.raw
raw/c4ca9e7b-5995-4db8-ab4a-5845c4808209.raw
raw/0854c5f7-f945-414c-870d-47046a1f6611.raw
raw/be0cfc04-1d28-427f-bbde-9e5ba1324cc4.raw
raw/b7e31f9c-a34a-499c-a15d-95497435c1cd.raw
raw/c1616a91-e9c9-4738-b61d-35cb7efbc7e8.raw
raw/bb7bad90-327a-4e96-80ce-1f1d48da13f8.raw
raw/00163764-bdd3-4a6b-a25e-af9c18c2eb39.raw
raw/5eecd45a-3a4a-493d-b1c0-d8f985ef8954.raw
raw/245892c8-68ff-41aa-8777-de9cbc6f9fd7.raw
raw/43f66bb5-ab58-40f0-8b70-4e1bba225fd1.raw
raw/50a91ac6-b9a8-4e94-ac78-57968e9cb8ae.raw
raw/aa2ff0bc-ec31-48eb-9bb8-8ea8bf99e63f.raw
raw/4ba20597-52a9-455a-bc5b-4047f4d46e93.raw
raw/2b282211-50

### Inspecting the INT8 DLC

After quantization, `qairt-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [8]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; optimization_pass_mode_config=; batch=None; squash_box_decoder=True; dump_exported_onnx=False; dumpIR=False; custom_op_config_paths=None; preprocess_roi_pool_inputs=True; saved_model_tag=serve; backend=HTP; preserve_io=[['layout']]; multi_time_steps_lstm=False; debug=-1; calc_static_encodings=False; dump_custom_io_config_template=; partial_graph_input_name=None; perform_sequence_construct_optimizer=False; adjust_nms_features_dims=True; user_custom_io=[]; disable_transform_tracking=False; disable_defer_loading=False; enable_Layout_Transform_v1=False; inject_cast_for_gather=True; converter_op_package_lib=; enable_experimental_feature=[]; handle_gather_negative_indices=True; disable_preserve_io=False; enable_tensor_deduplication=False; disable_batchnorm_folding=Fal

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.